# Full pipeline — neuro-symbolic verification for SimuHome QT4 scheduling

One notebook, one Kaggle session, run top to bottom:
1. Setup (clone SimuHome, install deps, write `src/`)
2. Z3 encoder sanity check (no GPU, no data, ~seconds)
3. Baseline reproduction (no GPU if using a hosted API; needs `OPENAI_API_KEY`)
4. Generate (query, IR) training data (no GPU)
5. LoRA fine-tune the intent parser (**needs GPU**)
6. Wire the fine-tuned parser + Z3 + verbalizer into `schedule_workflow`
7. Full evaluation (E1) + ablations

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **ON**. A T4 session is capped at ~9-12h continuous — if steps 5-7 don't fit in one sitting, just re-run this same notebook in a new session after 'Save Version': every expensive cell below checks for its own output first and skips itself if the work is already on disk, so nothing is repeated. Version `/kaggle/working` as a Kaggle Dataset between sessions so the skip-checks have something to find.

This replaces running `01`-`06` as six separate notebooks/sessions and manually passing Kaggle Datasets between them — same content, same tested `src/` modules, single linear run.


## 1. Setup


In [ ]:
!pip -q install -U z3-solver transformers peft accelerate bitsandbytes trl datasets torchao


In [ ]:
import os
# This kernel (parser, fine-tuning) may only use GPU 0. GPU 1 belongs to the vLLM server, and with both
# visible HF Trainer would wrap training in DataParallel and OOM there. Must be set before CUDA is first used.
# (The vLLM subprocess sets its own CUDA_VISIBLE_DEVICES, so it is unaffected.)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
PROJECT_ROOT = os.getcwd()  # capture once, before any cd -- reruns then can't nest
if not os.path.isdir('SimuHome'):
    !git clone --depth 1 https://github.com/holi-lab/SimuHome.git
# pyproject.toml pins requires-python >= 3.13; no 3.13-only syntax in the
# code (checked), so bypass the gate rather than fail on a 3.10/3.11 image.
!pip install -e ./SimuHome -q --ignore-requires-python
import sys
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'SimuHome'))


In [ ]:
import os; os.makedirs('irlib', exist_ok=True)


In [ ]:
%%writefile irlib/ir_schema.py
"""Typed intermediate representation (IR) for SimuHome QT4 scheduling requests.

An IR describes one `schedule_workflow`-shaped user request as a set of
target device assertions, each gated by one or more *time readings* (ways
the natural-language query expresses when the action should happen).
A feasible request has exactly one consistent time reading per goal.
An infeasible request states two readings that resolve to different times
for the same action (this is the "Contradiction Blindness" failure mode
SimuHome documents: the base agent never compares the readings).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Literal, Optional

Relation = Literal["after", "before", "at"]


@dataclass
class Anchor:
    """What a time offset is measured from."""

    kind: Literal["now", "device_event"] = "now"
    room_id: Optional[str] = None
    device_id: Optional[str] = None
    device_type: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "kind": self.kind,
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
        }


@dataclass
class TimeReading:
    """One way the query expresses *when* an action should occur.

    Exactly one of (offset_minutes with anchor) or (absolute_time) is the
    "primary" reading normally used to compute `at_minutes`. A second
    TimeReading on the same goal, disagreeing with the first once resolved
    to absolute minutes-since-base_time, is what makes a request infeasible.
    """

    relation: Relation
    anchor: Anchor
    offset_minutes: Optional[float] = None  # relative reading
    absolute_time: Optional[str] = None  # "HH:MM:SS", clock-time reading
    resolved_at_minutes: Optional[float] = None  # ground truth, minutes since base_time
    tolerance_minutes: float = 0.0

    def to_dict(self) -> dict[str, Any]:
        return {
            "relation": self.relation,
            "anchor": self.anchor.to_dict(),
            "offset_minutes": self.offset_minutes,
            "absolute_time": self.absolute_time,
            "resolved_at_minutes": self.resolved_at_minutes,
            "tolerance_minutes": self.tolerance_minutes,
        }


@dataclass
class Assertion:
    attribute: str
    value: Any
    value_type: Optional[str] = None
    description: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "attribute": self.attribute,
            "value": self.value,
            "value_type": self.value_type,
            "description": self.description,
        }


@dataclass
class Target:
    room_id: str
    device_id: str
    device_type: Optional[str]
    asserts: list[Assertion] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "asserts": [a.to_dict() for a in self.asserts],
        }


@dataclass
class Goal:
    """One scheduled effect: a set of device assertions gated by time reading(s)."""

    goal_id: int
    readings: list[TimeReading]
    targets: list[Target]
    # Set when the conflict is a duration/resource feasibility issue rather
    # than a stated-time mismatch (e.g. "finish washer exactly when
    # dishwasher finishes" but the washer cycle is longer). The Z3 encoder
    # cannot check these without a device duration model; flagged so callers
    # know to skip or handle them separately (see handoff Sec 6, open items).
    requires_duration_model: bool = False

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal_id": self.goal_id,
            "readings": [r.to_dict() for r in self.readings],
            "targets": [t.to_dict() for t in self.targets],
            "requires_duration_model": self.requires_duration_model,
        }


@dataclass
class IntentIR:
    query: str
    query_type: str
    case: Literal["feasible", "infeasible", "unknown"]
    base_time: str  # "YYYY-MM-DD HH:MM:SS"
    goals: list[Goal] = field(default_factory=list)
    conflict_type: Optional[str] = None  # ground-truth label, training-time only

    def to_dict(self) -> dict[str, Any]:
        return {
            "query": self.query,
            "query_type": self.query_type,
            "case": self.case,
            "base_time": self.base_time,
            "goals": [g.to_dict() for g in self.goals],
            "conflict_type": self.conflict_type,
        }


# ---------------------------------------------------------------------------
# Compact wire format (what the SLM is trained to emit)
# ---------------------------------------------------------------------------
# The full IntentIR.to_dict() is ~520 tokens: it echoes the query, repeats null
# fields, and includes labels the model must NOT see or produce (`case`,
# `conflict_type`) and simulator facts it cannot know (`base_time`, tolerances).
# The compact form keeps only what the query text determines: per goal, the time
# readings and the targets. Z3 decides feasibility; the model never says it.

DEFAULT_TOLERANCE_MINUTES = 0.5


def compact_ir_dict(ir_dict: dict, *, keep_resolved: bool = False) -> dict:
    """Full IR dict -> compact dict. Clock times are cut to HH:MM (the query
    states minutes, never the simulator's seconds)."""
    goals = []
    for g in ir_dict["goals"]:
        readings = []
        for r in g["readings"]:
            anchor = {"kind": r["anchor"]["kind"]}
            if r["anchor"].get("device_id"):
                anchor["device_id"] = r["anchor"]["device_id"]
            if r["anchor"].get("room_id"):
                anchor["room_id"] = r["anchor"]["room_id"]
            out = {"relation": r["relation"], "anchor": anchor}
            if r.get("offset_minutes") is not None:
                out["offset_minutes"] = r["offset_minutes"]
            if r.get("absolute_time") is not None:
                out["absolute_time"] = ":".join(r["absolute_time"].split(":")[:2])
            if keep_resolved and r.get("resolved_at_minutes") is not None:
                out["resolved_at_minutes"] = r["resolved_at_minutes"]
            readings.append(out)
        targets = [
            {
                "device_id": t["device_id"],
                "asserts": [{"attribute": a["attribute"], "value": a["value"]} for a in t.get("asserts", [])],
            }
            for t in g["targets"]
        ]
        goals.append({"readings": readings, "targets": targets})
    return {"goals": goals}


def ir_from_compact(compact: dict, query: str, base_time: str) -> "IntentIR":
    """Compact dict (+ the live query and simulator base_time) -> IntentIR."""
    goals = []
    for i, g in enumerate(compact["goals"]):
        readings = []
        for r in g["readings"]:
            a = r.get("anchor") or {}
            readings.append(
                TimeReading(
                    relation=r.get("relation", "after"),
                    anchor=Anchor(
                        kind=a.get("kind", "now"),
                        room_id=a.get("room_id"),
                        device_id=a.get("device_id"),
                    ),
                    offset_minutes=r.get("offset_minutes"),
                    absolute_time=r.get("absolute_time"),
                    resolved_at_minutes=r.get("resolved_at_minutes"),
                    tolerance_minutes=r.get("tolerance_minutes", DEFAULT_TOLERANCE_MINUTES),
                )
            )
        targets = [
            Target(
                room_id=t.get("room_id", ""),
                device_id=t["device_id"],
                device_type=None,
                asserts=[Assertion(attribute=a["attribute"], value=a["value"]) for a in t.get("asserts", [])],
            )
            for t in g.get("targets", [])
        ]
        goals.append(Goal(goal_id=i, readings=readings, targets=targets))
    return IntentIR(query=query, query_type="qt4", case="unknown", base_time=base_time, goals=goals)


In [ ]:
%%writefile irlib/constraint_encoder.py
"""Component 2: IR -> Z3 constraints over real-valued time, SAT/UNSAT check.

Time is encoded as a single Real variable per goal: minutes elapsed since
the episode's `base_time` ("now" = 0). Each TimeReading on a goal produces
one equality/inequality constraint on that variable. A goal is feasible iff
all its readings can be satisfied simultaneously (by the *same* time point,
within tolerance) — this is exactly the check SimuHome's `schedule_workflow`
never performs, and it is the "Contradiction Blindness" fix.

CPU-only, no GPU, no SLM required to run or test this module.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import z3

from irlib.ir_schema import Goal, IntentIR, TimeReading


def _parse_hms_to_minutes_of_day(hms: str) -> float:
    parts = [int(x) for x in hms.split(":")]
    h, m = parts[0], parts[1]
    s = parts[2] if len(parts) > 2 else 0
    return h * 60.0 + m + s / 60.0


def _is_minute_granular(hms: str) -> bool:
    return len(hms.split(":")) < 3


def _base_time_minutes_of_day(base_time: str) -> float:
    # base_time: "YYYY-MM-DD HH:MM:SS"
    hms = base_time.split(" ", 1)[1]
    return _parse_hms_to_minutes_of_day(hms)


def reading_to_absolute_minutes(reading: TimeReading, base_time: str) -> Optional[float]:
    """Resolve a single reading to minutes-since-base_time, when it is
    self-contained (i.e. an absolute clock-time reading anchored to "now").
    Device-event anchored readings can't be resolved without simulator
    state, so they stay symbolic in the Z3 encoding instead.
    """
    if reading.absolute_time is not None and reading.anchor.kind == "now":
        base_minutes_of_day = _base_time_minutes_of_day(base_time)
        target_minutes_of_day = _parse_hms_to_minutes_of_day(reading.absolute_time)
        delta = target_minutes_of_day - base_minutes_of_day
        # Clock times are same-day in SimuHome QT4 episodes; no midnight wrap handling needed.
        return delta
    return None


@dataclass
class GoalCheckResult:
    goal_id: int
    sat: bool
    reason: Optional[str] = None  # populated on UNSAT
    unsat_core: Optional[list[str]] = None
    resolved_minutes: Optional[float] = None  # populated on SAT


@dataclass
class FeasibilityResult:
    feasible: bool
    goal_results: list[GoalCheckResult]

    def explanation(self) -> str:
        bad = [g for g in self.goal_results if not g.sat]
        if not bad:
            return "All scheduling constraints are consistent."
        lines = [f"Goal {g.goal_id}: {g.reason}" for g in bad]
        return "Infeasible schedule — " + "; ".join(lines)


def _encode_goal(goal: Goal, base_time: str) -> GoalCheckResult:
    if goal.requires_duration_model:
        return GoalCheckResult(
            goal_id=goal.goal_id,
            sat=True,
            reason="skipped: requires a device duration model (not checked)",
        )

    solver = z3.Solver()
    t = z3.Real(f"t_goal_{goal.goal_id}")

    named_constraints: list[tuple[str, z3.BoolRef]] = []

    for i, reading in enumerate(goal.readings):
        extra = 1.0 if (reading.absolute_time and _is_minute_granular(reading.absolute_time)) else 0.0
        # An "HH:MM" clock time is only known to the minute, so widen by one minute.
        tol = z3.RealVal(reading.tolerance_minutes + extra)
        label = f"reading_{i}"

        if reading.anchor.kind == "device_event":
            # Anchor time is not known symbolically here (no simulator
            # state at IR-build time); readings against the SAME anchor
            # device are still comparable to each other via a shared
            # per-anchor symbolic variable.
            anchor_var = z3.Real(f"anchor_{goal.goal_id}_{reading.anchor.device_id}")
            if reading.resolved_at_minutes is not None:
                # Ground-truth / runtime-known anchor value (e.g. fetched by
                # the agent via get_attribute CountdownTime before invoking
                # the solver). Without this the anchor stays symbolic and a
                # device-event reading alone can never conflict with itself.
                anchor_label = f"{label}_anchor_known"
                solver.assert_and_track(
                    anchor_var == z3.RealVal(reading.resolved_at_minutes), z3.Bool(anchor_label)
                )
            if reading.offset_minutes is None:
                continue
            offset = z3.RealVal(reading.offset_minutes)
            if reading.relation == "after":
                target = anchor_var + offset
            elif reading.relation == "before":
                target = anchor_var - offset
            else:
                target = anchor_var
            named_constraints.append(
                (label, z3.And(t >= target - tol, t <= target + tol))
            )
        else:
            # "now"-anchored: either a relative offset from 0, or a stated
            # absolute clock time resolved to minutes-since-base_time.
            resolved = reading_to_absolute_minutes(reading, base_time)
            if resolved is not None:
                target = z3.RealVal(resolved)
            elif reading.offset_minutes is not None:
                offset = z3.RealVal(reading.offset_minutes)
                target = offset if reading.relation != "before" else -offset
            else:
                continue
            named_constraints.append(
                (label, z3.And(t >= target - tol, t <= target + tol))
            )

    if not named_constraints:
        return GoalCheckResult(goal_id=goal.goal_id, sat=True, reason=None)

    tracker_bools = []
    for label, constraint in named_constraints:
        b = z3.Bool(label)
        solver.assert_and_track(constraint, b)
        tracker_bools.append(b)

    result = solver.check()
    if result == z3.sat:
        model = solver.model()
        resolved_t = model.eval(t, model_completion=True)
        return GoalCheckResult(
            goal_id=goal.goal_id,
            sat=True,
            resolved_minutes=float(resolved_t.as_fraction()),
        )

    core = [str(c) for c in solver.unsat_core()]
    reading_summaries = {
        f"reading_{i}": _describe_reading(r) for i, r in enumerate(goal.readings)
    }
    # "*_anchor_known" trackers exist only to pin a symbolic anchor to a
    # runtime-known value; they aren't a distinct "reading" worth naming in
    # the human-facing explanation.
    described = [
        reading_summaries[c] for c in core if c in reading_summaries
    ]
    reason = " vs. ".join(described) if described else "unsatisfiable"
    return GoalCheckResult(goal_id=goal.goal_id, sat=False, reason=reason, unsat_core=core)


def _describe_reading(reading: TimeReading) -> str:
    if reading.absolute_time is not None:
        return f"stated clock time {reading.absolute_time}"
    if reading.offset_minutes is not None:
        anchor_desc = (
            "now"
            if reading.anchor.kind == "now"
            else f"{reading.anchor.device_id} finishing"
        )
        return f"{reading.offset_minutes} min {reading.relation} {anchor_desc}"
    return "unspecified time reading"


def check_feasibility(ir: IntentIR) -> FeasibilityResult:
    goal_results = [_encode_goal(goal, ir.base_time) for goal in ir.goals]
    feasible = all(g.sat for g in goal_results)
    return FeasibilityResult(feasible=feasible, goal_results=goal_results)


In [ ]:
%%writefile irlib/extract_ir_from_episode.py
"""Component: deterministic (query, IR) pair extraction from SimuHome episodes.

No LLM calls, no manual labeling. SimuHome benchmark/generated episodes
already carry, in `eval.goals` (ground-truth required behavior) and
`temporal_conflict` (ground-truth contradiction, infeasible episodes only),
every field needed to derive the IR mechanically. See handoff Sec 5.3.

Two IRs are produced per episode:
  - `ir_for_training`: what the SLM must learn to produce from the query
    text alone. Device-anchored readings never carry a numeric resolved
    time here, because the query text does not state one — the live
    simulator state does, and only the runtime agent can fetch it.
  - `debug_ground_truth`: the same IR with anchor times filled in from the
    simulator-computed fields, for exercising the Z3 encoder end-to-end in
    notebook 04 without needing a live simulator.
"""

from __future__ import annotations

import glob
import json
import os
from typing import Any, Optional

from irlib.ir_schema import Anchor, Assertion, Goal, IntentIR, Target, TimeReading, compact_ir_dict

# Conflict types where the contradiction is a stated-time mismatch (checkable
# from two TimeReadings alone). See handoff Sec 6: duration/resource
# feasibility conflicts (e.g. "finish washer exactly when dishwasher ends"
# but the washer cycle itself is longer) need a device duration model the
# Z3 encoder does not have yet — flagged, not silently mis-scored.
_DURATION_MODEL_CONFLICT_TYPES = {"completion_vs_pause"}


def _tolerance_minutes(when: dict[str, Any], tick_interval: float) -> float:
    return float(when.get("tolerance_ticks", 0)) * tick_interval / 60.0


def _goal_from_eval_goal(goal_id: int, raw_goal: dict[str, Any], tick_interval: float) -> Goal:
    when = raw_goal["when"]
    raw_anchor = raw_goal.get("anchor") or {}

    if raw_anchor.get("device_id"):
        anchor = Anchor(
            kind="device_event",
            room_id=raw_anchor.get("room_id"),
            device_id=raw_anchor.get("device_id"),
            device_type=raw_anchor.get("device_type"),
        )
    else:
        anchor = Anchor(kind="now")

    reading = TimeReading(
        relation=when.get("relation", "after"),
        anchor=anchor,
        offset_minutes=float(when.get("offset_minutes", 0.0)),
        tolerance_minutes=_tolerance_minutes(when, tick_interval),
    )

    targets = [
        Target(
            room_id=t["room_id"],
            device_id=t["device_id"],
            device_type=t.get("device_type"),
            asserts=[
                Assertion(
                    attribute=a["attribute"],
                    value=a["value"],
                    value_type=a.get("value_type"),
                    description=a.get("description"),
                )
                for a in t.get("asserts", [])
            ],
        )
        for t in raw_goal.get("targets", [])
    ]

    return Goal(goal_id=goal_id, readings=[reading], targets=targets)


def _inject_conflicting_reading(
    goal: Goal, temporal_conflict: dict[str, Any], for_training: bool
) -> Goal:
    """Add the second, contradicting TimeReading to the primary goal.

    Mutates and returns `goal`. If the conflict is duration/resource-based
    (no comparable second time reading exists), flags the goal instead.
    """
    conflict_type = temporal_conflict.get("type")

    if conflict_type in _DURATION_MODEL_CONFLICT_TYPES or "conflict_at" not in temporal_conflict:
        goal.requires_duration_model = True
        return goal

    anchor_end_at = temporal_conflict.get("anchor_end_at")
    expected_at = temporal_conflict.get("expected_at")
    conflict_time = temporal_conflict.get("conflict_time")
    relation = temporal_conflict.get("relation", "after")

    if anchor_end_at is not None:
        # Dependency-anchored primary reading (e.g. "20 min after dishwasher finishes").
        base_offset = temporal_conflict.get(
            "offset_minutes", temporal_conflict.get("delay_minutes", 0.0)
        )
        offset = float(base_offset)
        anchor = goal.readings[0].anchor
        if anchor.kind != "device_event":
            # impossible_early_end / absolute_time_mismatch only give
            # anchor_end_at without repeating anchor identity on every
            # field; goal.readings[0].anchor already carries it from
            # eval.goals, so this branch should not normally trigger.
            pass
        primary_reading = TimeReading(
            relation=relation,
            anchor=anchor,
            offset_minutes=offset,
            tolerance_minutes=goal.readings[0].tolerance_minutes,
            # resolved_at_minutes here is the ANCHOR EVENT's own resolved
            # time (e.g. when the dishwasher finishes), not the target time
            # — the Z3 encoder derives target = anchor +/- offset itself.
            resolved_at_minutes=None if for_training else float(anchor_end_at),
        )
        goal.readings[0] = primary_reading
    elif expected_at is not None:
        # Now-anchored primary reading (e.g. "13 minutes from now").
        goal.readings[0].offset_minutes = float(expected_at)
        goal.readings[0].anchor = Anchor(kind="now")

    if conflict_time is not None:
        goal.readings.append(
            TimeReading(
                relation="at",
                anchor=Anchor(kind="now"),
                absolute_time=conflict_time.split(" ")[-1] if " " in conflict_time else conflict_time,
                tolerance_minutes=goal.readings[0].tolerance_minutes,
            )
        )

    return goal


def extract_episode(episode: dict[str, Any], *, for_training: bool = True) -> IntentIR:
    """Build an IntentIR from one raw SimuHome benchmark/generated episode dict."""
    meta = episode["meta"]
    tick_interval = float(episode["initial_home_config"]["tick_interval"])
    base_time = episode["initial_home_config"]["base_time"]
    case = meta.get("case", "unknown")

    raw_goals = episode["eval"]["goals"]
    goals = [
        _goal_from_eval_goal(i, rg, tick_interval) for i, rg in enumerate(raw_goals)
    ]

    conflict_type = None
    if case == "infeasible" and "temporal_conflict" in episode:
        temporal_conflict = episode["temporal_conflict"]
        conflict_type = temporal_conflict.get("type")
        if goals:
            goals[0] = _inject_conflicting_reading(
                goals[0], temporal_conflict, for_training
            )

    return IntentIR(
        query=episode["query"],
        query_type=meta.get("query_type", "unknown"),
        case=case,
        base_time=base_time,
        goals=goals,
        conflict_type=conflict_type,
    )


def extract_pair(episode: dict[str, Any]) -> dict[str, Any]:
    """Return {"query", "ir_for_training", "debug_ground_truth", "meta"} for one episode."""
    training_ir = extract_episode(episode, for_training=True)
    debug_ir = extract_episode(episode, for_training=False)
    return {
        "query": episode["query"],
        "ir_for_training": compact_ir_dict(training_ir.to_dict()),
        "debug_ground_truth": debug_ir.to_dict(),
        "meta": episode["meta"],
    }


def extract_dataset(benchmark_dir: str, pattern: str = "qt4-*_seed_*.json") -> list[dict[str, Any]]:
    """Extract (query, IR) pairs for every QT4 episode file in `benchmark_dir`."""
    # Benchmark files sit directly in the dir; generator output is <run>/episodes/*.json.
    paths = sorted(
        glob.glob(os.path.join(benchmark_dir, "qt4-*.json"))
        + glob.glob(os.path.join(benchmark_dir, "*", "episodes", "qt4-*.json"))
    )
    pairs = []
    errors = []
    for path in paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                episode = json.load(f)
            pair = extract_pair(episode)
            pair["source_file"] = os.path.basename(path)
            pairs.append(pair)
        except Exception as e:  # keep going; report at the end
            errors.append((path, str(e)))
    if errors:
        print(f"[extract_dataset] {len(errors)} file(s) failed to extract:")
        for path, err in errors[:20]:
            print(f"  {os.path.basename(path)}: {err}")
    return pairs


if __name__ == "__main__":
    import sys

    bench_dir = sys.argv[1] if len(sys.argv) > 1 else "SimuHome/data/benchmark"
    out_path = sys.argv[2] if len(sys.argv) > 2 else "data/qt4_ir_pairs.jsonl"
    pairs = extract_dataset(bench_dir)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    print(f"Extracted {len(pairs)} pairs -> {out_path}")


In [ ]:
open('irlib/__init__.py', 'a').close()
import sys
sys.path.insert(0, '.')


## 2. Z3 encoder sanity check

Hand-crafted QT4-1/2/3 feasible + infeasible examples, checked before any model is involved (per the project handoff: never blame the SLM for a bad encoding). Locally, this scores **286/286** against all 300 downloaded real QT4 benchmark episodes (150 feasible + 136 infeasible correct, 14 duration-model conflicts correctly flagged as out of scope — see the README's Known limitation section).


In [ ]:
from irlib.ir_schema import IntentIR, Goal, TimeReading, Anchor, Target, Assertion
from irlib.constraint_encoder import check_feasibility

# QT4-1 infeasible: stated clock time contradicts stated relative offset.
ir_infeasible = IntentIR(
    query='At 11:35 AM, that is 13 minutes from now, turn on living room light 1.',
    query_type='qt4-1', case='infeasible', base_time='2025-08-23 11:36:54',
    goals=[Goal(goal_id=0, readings=[
        TimeReading(relation='after', anchor=Anchor(kind='now'), offset_minutes=13, tolerance_minutes=5),
        TimeReading(relation='at', anchor=Anchor(kind='now'), absolute_time='11:35:54', tolerance_minutes=5),
    ], targets=[Target(room_id='living_room', device_id='living_room_dimmable_light_1', device_type='dimmable_light',
                        asserts=[Assertion(attribute='1.OnOff.OnOff', value=True)])])],
)
result = check_feasibility(ir_infeasible)
print(result.feasible, result.explanation())
assert not result.feasible


In [ ]:
from irlib.extract_ir_from_episode import extract_dataset

pairs = extract_dataset('SimuHome/data/benchmark')
counts = {'feasible_correct': 0, 'feasible_wrong': 0, 'infeasible_correct': 0, 'infeasible_wrong': 0, 'skipped': 0}
for pair in pairs:
    d = pair['debug_ground_truth']
    goals = []
    for g in d['goals']:
        readings = [TimeReading(anchor=Anchor(**r['anchor']), **{k: v for k, v in r.items() if k != 'anchor'}) for r in g['readings']]
        targets = [Target(room_id=t['room_id'], device_id=t['device_id'], device_type=t['device_type'],
                           asserts=[Assertion(**a) for a in t['asserts']]) for t in g['targets']]
        goals.append(Goal(goal_id=g['goal_id'], readings=readings, targets=targets, requires_duration_model=g['requires_duration_model']))
    ir = IntentIR(query=d['query'], query_type=d['query_type'], case=d['case'], base_time=d['base_time'], goals=goals, conflict_type=d['conflict_type'])
    r = check_feasibility(ir)
    expected = ir.case == 'feasible'
    if any('duration model' in (g.reason or '') for g in r.goal_results) and not expected:
        counts['skipped'] += 1
        continue
    key = ('feasible' if expected else 'infeasible') + ('_correct' if r.feasible == expected else '_wrong')
    counts[key] += 1
print(counts)
assert counts['feasible_wrong'] == 0 and counts['infeasible_wrong'] == 0


## 3. Baseline reproduction (agent LLM: a hosted API, NVIDIA Gemma by default)

Sanity-check SimuHome's own ReAct baseline before building anything on top. The agent LLM is a **hosted API called through its OpenAI-compatible endpoint**. The default is `google/gemma-4-31b-it` on NVIDIA's free hosted API (the only strong-ish model that still worked when tested; Llama-3.3-70B and most others there have been retired, and free hosted models can disappear, so record the exact model id and the dates you ran). Free tiers are rate- and quota-limited, so this notebook:
- throttles every LLM call (`MIN_SECONDS_BETWEEN_CALLS`) and counts them,
- runs the evaluator **in this process** for BOTH arms, so the throttle, the counter and the request fixes apply to both,
- runs E1 a few (sub-type, case) cells at a time (`E1_CELLS`) and saves after each run, so you can spread it over several days.

Get a key at https://build.nvidia.com (Get API Key), add it as the Kaggle secret `NVIDIA_API_KEY`. To use another provider (Gemini, Mistral, Groq...), change the four lines marked below in the backend cell: base URL, model id, secret name, and the throttle.


In [ ]:
import os, sys, time, subprocess, requests

LLM_BACKEND = 'custom'   # 'custom' (any OpenAI-compatible API; default = Gemini) | 'local_vllm' | 'openrouter'

if LLM_BACKEND == 'custom':
    LLM_API_BASE = 'https://integrate.api.nvidia.com/v1'    # Gemini alt: 'https://generativelanguage.googleapis.com/v1beta/openai'
    LLM_MODEL = 'google/gemma-4-31b-it'         # the only NVIDIA-hosted model that passed our agent-format test; 'gemini-2.5-flash' for Gemini
    LLM_API_KEY_ENV = 'NVIDIA_API_KEY'          # name of the Kaggle secret that holds your key
    from kaggle_secrets import UserSecretsClient
    os.environ[LLM_API_KEY_ENV] = UserSecretsClient().get_secret(LLM_API_KEY_ENV)
elif LLM_BACKEND == 'openrouter':
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    except Exception as e:
        print('Set OPENROUTER_API_KEY manually:', e)
    LLM_MODEL = 'meta-llama/llama-3.3-70b-instruct'
    LLM_API_BASE = 'https://openrouter.ai/api/v1'
    LLM_API_KEY_ENV = 'OPENROUTER_API_KEY'
elif LLM_BACKEND == 'local_vllm':
    LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
    LLM_API_BASE = 'http://127.0.0.1:8001/v1'
    LLM_API_KEY_ENV = None
else:
    raise ValueError(f'unknown LLM_BACKEND: {LLM_BACKEND!r}')

# ---- free-tier guards, applied by the shim in the next cell ----
MIN_SECONDS_BETWEEN_CALLS = 2.0   # NVIDIA free tier is 40 requests/min (1.5 s); raise it if you see 429s
LLM_CALL_CAP = 1500              # hard stop: no more than this many LLM requests in one session (a run is about 80; the 7-run plan is about 600)
DROP_PARAMS = ['seed', 'response_format']   # NVIDIA lists structured output as unsupported; the agent's JSON is then parsed from text
STRIP_SCHEMA_STRICT = True        # drop `strict` / `additionalProperties` from the JSON schema (Gemini accepts a subset)
EXTRA_KWARGS = {}                 # e.g. {'reasoning_effort': 'low'} to cut Gemini thinking time/tokens
if LLM_BACKEND == 'local_vllm':
    MIN_SECONDS_BETWEEN_CALLS, DROP_PARAMS, STRIP_SCHEMA_STRICT = 0.0, [], False

vllm_proc = None
if LLM_BACKEND == 'local_vllm':
    VLLM_PY = '/tmp/vllm_env/bin/python'
    # Clean env for the venv: Kaggle's PYTHONPATH/sitecustomize import packages that don't exist in the venv.
    clean_env = {k: v for k, v in os.environ.items() if k != 'PYTHONPATH'}
    clean_env['PYTHONNOUSERSITE'] = '1'
    def _vllm_ok():
        if not os.path.exists(VLLM_PY): return False
        return subprocess.run([VLLM_PY, '-c', 'import vllm'], capture_output=True, env=clean_env).returncode == 0
    if not _vllm_ok():
        # uv creates the venv (with or without pip) and installs much faster than pip.
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '/tmp/vllm_env', '--python', sys.executable, '--clear'], check=True, env=clean_env)
        r = subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', VLLM_PY, 'vllm'],
                           capture_output=True, text=True, env=clean_env)
        print(r.stdout[-800:], r.stderr[-2000:])
        if not _vllm_ok():
            raise RuntimeError('vLLM is still not importable in /tmp/vllm_env (install output above)')
    print('vLLM importable in the isolated venv')
    n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines())
    env = dict(clean_env)
    if n_gpu >= 2:
        env['CUDA_VISIBLE_DEVICES'] = '1'; util = '0.88'   # leave GPU 0 for the parser / fine-tuning
    else:
        util = '0.45'
    print(f'{n_gpu} GPU(s); starting vLLM with gpu-memory-utilization={util}')
    log = open('/tmp/vllm_server.log', 'w')
    vllm_proc = subprocess.Popen(
        [VLLM_PY, '-m', 'vllm.entrypoints.openai.api_server', '--model', LLM_MODEL, '--served-model-name', LLM_MODEL,
         '--port', '8001', '--dtype', 'half', '--max-model-len', '8192', '--gpu-memory-utilization', util],
        env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(450):   # up to 15 min (first run downloads the weights)
        if vllm_proc.poll() is not None:
            print(open('/tmp/vllm_server.log').read()[-3000:]); raise RuntimeError('vLLM exited early (log above)')
        try:
            requests.get('http://127.0.0.1:8001/v1/models', timeout=1); print('local vLLM server up'); break
        except Exception:
            time.sleep(2)
    else:
        print(open('/tmp/vllm_server.log').read()[-3000:]); raise RuntimeError('vLLM did not come up in 15 min')


### Request shim (free-tier guards)

SimuHome builds its own OpenAI client, so we patch the SDK's `chat.completions.create` once, in this process: it spaces calls out, counts them, drops parameters the provider rejects, and relaxes the JSON schema for providers that accept only a subset. It affects the baseline AND verified arms identically.


In [ ]:
import threading, copy
from openai.resources.chat.completions import Completions
import openai
SHIM_RETRIES = 4   # extra attempts on provider errors, with backoff (so a brief 504 does not kill an unattended run)
_RETRYABLE = (openai.InternalServerError, openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError)

LLM_CALLS = {'count': 0, 'last': 0.0}
_llm_lock = threading.Lock()
_orig_create = getattr(Completions.create, '_orig', Completions.create)   # idempotent if this cell is re-run

def _strip_schema(node):
    if isinstance(node, dict):
        node.pop('additionalProperties', None)
        for v in node.values():
            _strip_schema(v)
    elif isinstance(node, list):
        for v in node:
            _strip_schema(v)

def _shimmed_create(self, *args, **kwargs):
    for k in DROP_PARAMS:
        kwargs.pop(k, None)
    kwargs.update(EXTRA_KWARGS)
    rf = kwargs.get('response_format')
    if STRIP_SCHEMA_STRICT and isinstance(rf, dict) and rf.get('type') == 'json_schema':
        rf = copy.deepcopy(rf)
        rf['json_schema'].pop('strict', None)
        _strip_schema(rf['json_schema'].get('schema', {}))
        kwargs['response_format'] = rf
    _cap = globals().get('LLM_CALL_CAP')
    for attempt in range(SHIM_RETRIES + 1):
        with _llm_lock:
            if _cap and LLM_CALLS['count'] >= _cap:
                raise RuntimeError(f'LLM call cap reached ({_cap}); raise LLM_CALL_CAP if you really want more')
            wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - LLM_CALLS['last'])
            if wait > 0:
                time.sleep(wait)
            LLM_CALLS['last'] = time.time()
            LLM_CALLS['count'] += 1          # every attempt counts toward the cap, retries included
        try:
            return _orig_create(self, *args, **kwargs)
        except _RETRYABLE as e:              # provider-side trouble (504/5xx, 429, timeouts, dropped connections)
            if attempt == SHIM_RETRIES:
                raise
            delay = 20 * 2 ** attempt        # 20, 40, 80, 160 s
            print(f'[shim] {type(e).__name__}: retry {attempt + 1}/{SHIM_RETRIES} in {delay}s', flush=True)
            time.sleep(delay)

_shimmed_create._orig = _orig_create
Completions.create = _shimmed_create
print('request shim installed; min seconds between calls =', MIN_SECONDS_BETWEEN_CALLS)


### Preflight: prove the LLM answers BEFORE spending quota on it

Sends the request exactly the way SimuHome's ReAct agent does (temperature 0, seed, strict JSON-schema output with its `thought/action/action_input` schema), through the shim above. It **stops** with the provider's raw error if it fails. Common causes: quota exhausted (429), wrong model id (404), bad key (401/403).


In [ ]:
import json
from openai import OpenAI

STOP_ON_BAD_FORMAT = False   # the probe below uses a toy prompt, so it is only a hint; the real check is the agent smoke test in the E1 cell

def llm_preflight():
    client = OpenAI(api_key=(os.environ.get(LLM_API_KEY_ENV, '') if LLM_API_KEY_ENV else 'dummy'), base_url=LLM_API_BASE)
    schema = {'type': 'json_schema', 'json_schema': {'name': 'react_response', 'strict': True, 'schema': {
        'type': 'object',
        'properties': {'thought': {'type': 'string'}, 'action': {'type': 'string'}, 'action_input': {'type': 'string', 'default': '{}'}},
        'required': ['thought', 'action', 'action_input'], 'additionalProperties': False}}}
    msgs = [{'role': 'user', 'content': 'Use the tool get_current_time. Answer with thought, action and action_input (a JSON string).'}]
    for label, extra in [('plain', {}), ('json_schema, as the ReAct agent sends it', {'response_format': schema})]:
        try:
            r = client.chat.completions.create(model=LLM_MODEL, temperature=0.0, seed=42, messages=msgs, **extra)
        except Exception as e:
            print(f'[{label}] FAILED: {repr(e)[:900]}')
            raise RuntimeError(f'LLM preflight failed ({label}); see the provider error above')
        txt = r.choices[0].message.content or ''
        print(f'[{label}] OK: {txt[:220]!r}')
        if extra:   # the agent-format check: the reply must parse as {thought, action, action_input}
            try:
                d = json.loads(txt[txt.find('{'):txt.rfind('}') + 1])
                fmt_ok = {'thought', 'action', 'action_input'} <= set(d)
            except Exception:
                fmt_ok = False
            print('   agent JSON format (thought/action/action_input):', 'OK' if fmt_ok else 'NOT PARSEABLE with this toy prompt (only a hint; the agent smoke test before E1 is the real check)')
            if not fmt_ok and STOP_ON_BAD_FORMAT:
                raise RuntimeError('agent model did not return the ReAct JSON format on the probe; change LLM_MODEL (or set STOP_ON_BAD_FORMAT=False)')
    print('LLM preflight OK; calls used so far:', LLM_CALLS['count'])

llm_preflight()


In [ ]:
import subprocess, time, requests
sim_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.simulator.api.app:app', '--port', '8000'],
    cwd='SimuHome', stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        requests.get('http://127.0.0.1:8000/docs', timeout=1)
        print('simulator up')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('simulator failed to start; check sim_proc.stdout')


### Run SimuHome's evaluator in this process

`main.py eval-start` runs the evaluator in a fresh subprocess, where none of the guards above (or the verifier patch in section 6) would apply. So both arms call `parallel_model_evaluation.main()` directly. `max_workers` is 1: the shim serialises calls anyway and a free tier can't use more.


In [ ]:
import os, sys, json, yaml, importlib, shutil
os.chdir(os.path.join(PROJECT_ROOT, 'SimuHome'))

def build_eval_spec(run_id, qt, case, seed_range):
    api_key_value = f'env:{LLM_API_KEY_ENV}' if LLM_API_KEY_ENV else None
    return {
        'schema': 'simuhome-eval-spec-v1',
        'api': {'base': LLM_API_BASE, 'key': api_key_value},
        'run': {'id': run_id, 'output_root': 'experiments'},
        'episode': {'dir': 'data/benchmark', 'qt': qt, 'case': case, 'seed': seed_range},
        'strategy': {'name': 'react', 'timeout': 120, 'temperature': 0.0, 'max_steps': 20},
        'orchestration': {'max_workers': 1, 'simulator_start_timeout': 30,
                          'simulator_start_retries': 1, 'evaluation_retries': 1,
                          'allow_partial_start': True},
        'judge': {'model': LLM_MODEL, 'api_base': LLM_API_BASE, 'api_key': api_key_value},
        'models': [{'model': LLM_MODEL, 'api_base': LLM_API_BASE, 'api_key': api_key_value}],
    }

import types, multiprocessing

class _InlinePool:
    """SimuHome's evaluator runs each model's episodes in an mp.Pool of SEPARATE worker processes. Anything patched in
    this notebook (the request shim, the verifier, their counters, the GPU-resident parser) would not exist there, and
    on Linux forked workers cannot use the CUDA context at all. This drop-in runs the same work in THIS process."""
    def __init__(self, processes=None, initializer=None, initargs=()):
        # The worker initializer points sys.stdout/sys.stderr at /dev/null (fine in a worker process, but here it would
        # silence this notebook), so remember them and put them back when the pool is closed.
        self._saved = (sys.stdout, sys.stderr)
        if initializer:
            initializer(*initargs)
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        sys.stdout, sys.stderr = self._saved
        return False
    def map_async(self, fn, iterable):
        results = [fn(x) for x in iterable]
        class _Done:
            def ready(self): return True
            def get(self): return results
        return _Done()

def run_inproc(run_id, qt, case, seed_range):
    spec_path = f'{run_id}.yaml'
    shutil.rmtree(f'experiments/{run_id}', ignore_errors=True)   # SimuHome refuses an existing run folder; a failed earlier attempt leaves one
    with open(spec_path, 'w') as f:
        yaml.safe_dump(build_eval_spec(run_id, qt, case, seed_range), f, sort_keys=False)
    pme = importlib.import_module('src.cli.parallel_model_evaluation')
    pme.mp = types.SimpleNamespace(Queue=multiprocessing.Queue, Pool=_InlinePool)   # see _InlinePool above
    old_argv = sys.argv
    sys.argv = ['parallel_model_evaluation', '--spec', spec_path]
    try:
        pme.main()
    except SystemExit as e:
        print('evaluator exited with code', e.code)
    finally:
        sys.argv = old_argv
    if not os.path.exists(f'experiments/{run_id}/run_summary.json'):
        raise RuntimeError(f'no run_summary.json for {run_id}; read the evaluator output above')

def aggregate_run(run_id):
    # Per-episode results live in a per-model SUBFOLDER, and the aggregator only reads *.json directly inside the
    # folder it is given, so aggregate each subfolder (the run folder itself yields 0 episodes).
    base = f'experiments/{run_id}'
    if not os.path.isdir(base):
        return {}
    out = {}
    for sub in sorted(d for d in os.listdir(base) if os.path.isdir(f'{base}/{d}')):
        subprocess.run([sys.executable, '-m', 'src.cli.main', 'aggregate', '--dir', f'{base}/{sub}'], capture_output=True, text=True)
        f = f'{base}/{sub}/aggregate_summary.json'
        if os.path.exists(f):
            out[sub] = json.load(open(f))
    return out


### Baseline sanity check (5 episodes, about 50 LLM calls)

Read `infra_errors` first: above 0 means the API failed (quota, 429, rejected request), which says nothing about the agent. A healthy result is infra_errors 0 and an accuracy well above 0. If accuracy is 0 with `schema_errors` > 0, the model cannot produce SimuHome's JSON action format.


In [ ]:
RUN_BASELINE_SMOKE = False   # True only on a first, attended session (about 15-30 min); the preflight already checks the API
BASELINE_SEEDS = '1 - 3'   # about 30 LLM calls; raise once you know your quota
if RUN_BASELINE_SMOKE and not os.path.exists('experiments/baseline_qt4_1_feasible/run_summary.json'):
    run_inproc('baseline_qt4_1_feasible', 'qt4-1', 'feasible', BASELINE_SEEDS)
for model_dir, agg in aggregate_run('baseline_qt4_1_feasible').items():
    print(model_dir, json.dumps(agg['overall'], indent=2))
print('LLM calls used so far:', LLM_CALLS['count'])
os.chdir(PROJECT_ROOT)


### Baseline check: is the agent working at all?

The numbers above only say how many episodes passed. This cell explains *why*, using the per-episode result files. Run it after the baseline (and reuse `inspect_run(run_id)` later on any E1 run).

**How to read it**

| What you see | What it means | What to do |
|---|---|---|
| `infra_errors` > 0 | The API or simulator failed (quota, 429/5xx, retired model). The accuracy is **not meaningful**. | Read the provider message, fix quota or model, then delete `experiments/<run>` and rerun. |
| `schema_errors` > 0 | The model broke SimuHome's JSON action format (`thought` / `action` / `action_input`). | A stronger model is needed. |
| accuracy 0, no errors | It runs but solves nothing. Look at the per-episode lines. | See the list below. |
| accuracy > 0, no errors | The agent works. | Proceed to E1, but 3 episodes is a tiny sample. |

**Per-episode lines:** `required lookups 0/2` means it never called `get_room_devices`, which the scorer requires (and usually means it invented device ids). An `actions` list ending in `finish` with no `schedule_workflow` means it answered without scheduling anything. A `schedule_workflow` with the wrong time means a time-arithmetic error.


In [ ]:
import glob, json, os
os.chdir(os.path.join(PROJECT_ROOT, 'SimuHome'))

def inspect_run(run_id, show=10):
    base = f'experiments/{run_id}'
    runs = aggregate_run(run_id)
    if not runs:
        print(f'{run_id}: no per-model results found under {base}'); return
    for model_dir, agg in runs.items():
        o = agg['overall']
        print(f'== {run_id}   (model folder: {model_dir})')
        print(f"   episodes evaluated: {o['evaluated_total']}/{o['total']}    correct: {o['correct']}    accuracy: {o['accuracy']:.2f}")
        print(f"   infra_errors (API/simulator failed): {o['infra_errors']}    schema_errors (bad JSON action format): {o['schema_errors']}")
        print(f"   avg seconds per episode: {o['avg_duration_sec']}    avg tool calls per episode: {o['avg_actions']}")
        notes = []
        if o['evaluated_total'] == 0:
            notes.append('NO episodes were evaluated: read the evaluator output above for the real error.')
        if o['infra_errors']:
            notes.append('API/simulator errors: check your quota and the provider message. The accuracy is NOT meaningful yet.')
        if o['schema_errors']:
            notes.append('The model broke the JSON action format in some episodes (a stronger model is needed).')
        if not notes and o['accuracy'] == 0:
            notes.append('Runs mechanically but solved nothing: see the per-episode lines (skipped lookups? invented device ids? wrong times?).')
        if not notes:
            notes.append('GO: the agent works. With only a few episodes treat this as a smoke test, not a result.')
        for n in notes:
            print('   ->', n)
        files = [f for f in sorted(glob.glob(f'{base}/{model_dir}/*.json')) if not f.endswith('aggregate_summary.json')]
        print('   per-episode:')
        for f in files[:show]:
            d = json.load(open(f))
            ev = d.get('evaluation_result') or {}
            steps = d.get('steps') or []
            actions = [s.get('action') for s in steps]
            req = ev.get('required_actions') or []
            req_txt = f"{sum(1 for r in req if r.get('invoked'))}/{len(req)}" if req else 'n/a'
            answer = ''
            for s in steps:
                if s.get('action') == 'finish':
                    ai = s.get('action_input')
                    answer = (ai.get('answer') if isinstance(ai, dict) else str(ai))[:90]
            print(f"   seed {d.get('seed')}: score={ev.get('score')}  error={ev.get('error_type') or '-'}  required lookups {req_txt}")
            print(f'            actions: {actions}')
            if answer:
                print(f'            final answer: {answer!r}')
        print()

inspect_run('baseline_qt4_1_feasible')
print('LLM calls used so far:', LLM_CALLS['count'])
os.chdir(PROJECT_ROOT)


## 4. Generate (query, IR) training data

**What this section does:** it builds the *training data* for the intent parser (the small model trained in section 5). It does not train anything itself.

**How the labels are made (no LLM, no manual work):** `src/extract_ir_from_episode.py` reads the answer key already stored in each SimuHome episode (`eval.goals`, and `temporal_conflict` for infeasible ones) and writes the target IR. Each training pair is (request sentence, IR).

**Which episodes are used (current setting):** the 300 native QT4 benchmark episodes, split by seed: every episode whose seed is divisible by 5 (54) is **held out** for testing; the rest are split 90/10 into train (221) and validation (25). Generating extra synthetic episodes is possible (`RUN_GENERATION = True`; it needs an LLM to write the requests) but is **off**.

# episode.home is REQUIRED by SimuHome's generator (the earlier run died with
# 'episode.home must be a mapping'); this block is copied from gen_spec.example.yaml.
HOME_SCHEMA = {'room_count': 5, 'devices_per_room': {'min': 4, 'max': 7},
               'environment': {'temperature_c': {'min': 22, 'max': 32}, 'humidity_pct': {'min': 35, 'max': 65},
                               'illuminance_lux': {'min': 100, 'max': 1500}, 'pm10_ugm3': {'min': 20, 'max': 100}}}
RUN_GENERATION = False   # True only to build a larger training set; then delete data/qt4_ir_v2_* and the v2 adapter to retrain
GEN_SEEDS = '1000-1049'   # 50 per (sub-type, case) x 6 = 300 episodes; widen once the free API proves fast enough
api_key_value = f'env:{LLM_API_KEY_ENV}' if LLM_API_KEY_ENV else None

for qt in ['qt4-1', 'qt4-2', 'qt4-3']:
    for case in ['feasible', 'infeasible']:
        if not RUN_GENERATION:
            continue
        run_id = f'gen_{qt}_{case}'.replace('-', '_')
        episodes_dir = f'../data/generated_qt4/{run_id}/episodes'
        if glob.glob(f'{episodes_dir}/*.json'):
            print('skip (already generated):', run_id)
            continue
        spec = {'schema': 'simuhome-gen-spec-v1',
                'run': {'id': run_id, 'output_root': '../data/generated_qt4'},
                'episode': {'qt': qt, 'case': case, 'seed': GEN_SEEDS, 'base_date': '2025-08-23', 'home': HOME_SCHEMA},
                'llm': {'model': LLM_MODEL, 'api_base': LLM_API_BASE, 'api_key': api_key_value, 'temperature': 1}}
        spec_path = f'{run_id}.yaml'
        with open(spec_path, 'w') as f:
            yaml.safe_dump(spec, f, sort_keys=False)
        proc = subprocess.run([sys.executable, '-m', 'src.cli.episode_generator', '--spec', spec_path],
                              capture_output=True, text=True)
        n = len(glob.glob(f'{episodes_dir}/*.json'))
        print(f'{run_id}: exit={proc.returncode} episodes_written={n}')
        if proc.returncode != 0 or n == 0:
            # Real cause is recorded per seed in run_state.json (the console tail is only progress bars).
            state_path = f'../data/generated_qt4/{run_id}/run_state.json'
            if os.path.exists(state_path):
                seeds_state = json.load(open(state_path)).get('seeds', {})
                errs = collections.Counter((v.get('error') or '')[:300] for v in seeds_state.values() if v.get('status') == 'failed')
                for msg, cnt in errs.most_common(3):
                    print(f'   {cnt} seeds failed with: {msg}')
            else:
                print('   no run_state.json written; console tail:', [l for l in (proc.stdout + proc.stderr).replace(chr(13), chr(10)).splitlines() if l.strip() and '%|' not in l][-8:])
os.chdir(PROJECT_ROOT)


In [ ]:
import json, os, random

if not os.path.exists('data/qt4_ir_v2_train.jsonl'):
    native = extract_dataset('SimuHome/data/benchmark')
    generated = extract_dataset('data/generated_qt4') if os.path.isdir('data/generated_qt4') else []
    print(f'native={len(native)} generated={len(generated)}')

    if generated:
        # Clean split: train/val come ONLY from generated episodes; ALL native episodes are test.
        trainable, held_out_test = generated, native
    else:
        # Generation produced nothing. Do NOT train on the whole native set (that contaminates E1).
        # Hold out every 5th seed of the native set as test and train only on the rest.
        print('WARNING: no generated episodes -- falling back to a seed-based split of the NATIVE set. '
              'This is a small, weaker setup; fix generation (see the cell above) for the real run.')
        held_out_test = [p for p in native if int(p['meta']['seed']) % 5 == 0]
        trainable = [p for p in native if int(p['meta']['seed']) % 5 != 0]

    random.seed(0)
    random.shuffle(trainable)
    n = len(trainable)
    train, val = trainable[: int(n * 0.9)], trainable[int(n * 0.9):]
    overlap = {p['query'] for p in train + val} & {p['query'] for p in held_out_test}
    assert not overlap, f'{len(overlap)} test queries also appear in train/val -- split is contaminated'

    os.makedirs('data', exist_ok=True)
    for split_name, split_data in [('train', train), ('val', val), ('test_held_out', held_out_test)]:
        with open(f'data/qt4_ir_v2_{split_name}.jsonl', 'w', encoding='utf-8') as f:
            for row in split_data:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'train={len(train)} val={len(val)} held_out_test={len(held_out_test)}')
else:
    print('data/qt4_ir_v2_train.jsonl already exists, skipping extraction')


## 5. LoRA fine-tune the intent parser

**Open items — decide from this run, don't guess upfront:** start with Qwen3-1.7B (fits a T4 with LoRA + bf16); move to 4B only if 1.7B underfits the IR-JSON task. Checkpointing is mandatory, not optional, per the handoff — a session can be killed mid-run with no warning.


In [ ]:
import json

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_rows = load_jsonl('data/qt4_ir_v2_train.jsonl')
val_rows = load_jsonl('data/qt4_ir_v2_val.jsonl')
print(len(train_rows), 'train', len(val_rows), 'val')


In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    'You convert a smart-home scheduling request into a strict JSON '
    'intermediate representation (IR). Output ONLY the JSON object, '
    'no prose, no code fences.'
)

def format_example(row):
    target = json.dumps(row['ir_for_training'], ensure_ascii=False)
    return {'text': f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{row['query']}\n<|assistant|>\n{target}"}

train_ds = Dataset.from_list([format_example(r) for r in train_rows])
val_ds = Dataset.from_list([format_example(r) for r in val_rows])


In [ ]:
MODEL_NAME = 'Qwen/Qwen3-1.7B'  # bump to Qwen/Qwen3-4B only if this underfits
ADAPTER_DIR = 'qwen3_ir_parser_adapter_v2'

import torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Reuse a previously trained adapter if you attached one as a Kaggle Dataset (skips retraining).
import glob, shutil
if not os.path.isdir(ADAPTER_DIR):
    for cand in glob.glob('/kaggle/input/*/qwen3_ir_parser_adapter_v2') + glob.glob('/kaggle/input/*/*/qwen3_ir_parser_adapter_v2'):
        shutil.copytree(cand, ADAPTER_DIR)
        print('reusing adapter from', cand)
        break

if os.path.isdir(ADAPTER_DIR):
    print(f'{ADAPTER_DIR} already exists -- loading the fine-tuned adapter instead of retraining')
    from peft import PeftModel
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map={'': 0})
    intent_parser = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
else:
    intent_parser = None  # trained in the next cell


In [ ]:
if intent_parser is None:
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer, SFTConfig
    import glob

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map={'': 0})
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    # NOT pre-wrapped with get_peft_model: TRL applies LoRA itself via peft_config. Pre-wrapping makes
    # model.forward a functools.partial and trl's chunked-loss patch then crashes on `__func__`.

    import inspect
    OUT_DIR = 'checkpoints_v2'
    # trl renamed max_seq_length -> max_length in newer releases; use whichever this install has.
    _len_kw = 'max_length' if 'max_length' in inspect.signature(SFTConfig.__init__).parameters else 'max_seq_length'
    sft_config = SFTConfig(
        output_dir=OUT_DIR, per_device_train_batch_size=4, gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=2e-4, logging_steps=10,
        save_strategy='steps', save_steps=14, save_total_limit=3,
        eval_strategy='steps', eval_steps=14, bf16=True, report_to=[],
        dataset_text_field='text', **{_len_kw: 1024},
        loss_type='nll',  # trl's default 'chunked_nll' patches model.forward and crashes here
    )
    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds, eval_dataset=val_ds, peft_config=lora_config)

    existing_ckpts = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'))
    resume_from = existing_ckpts[-1] if existing_ckpts else None
    print('Resuming from', resume_from) if resume_from else print('Starting fresh run')
    trainer.train(resume_from_checkpoint=resume_from)
    json.dump(trainer.state.log_history, open('train_log.json', 'w'), indent=2)   # per-step loss / eval loss / token accuracy, for the paper

    model = trainer.model   # the LoRA-wrapped model (the local `model` is still the bare base model)
    model.print_trainable_parameters()
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    intent_parser = model
    print('Saved adapter to', ADAPTER_DIR)
intent_parser.eval()


In [ ]:
def parse_intent(query: str, max_new_tokens=700) -> dict:
    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{query}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(intent_parser.device)
    out = intent_parser.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return json.loads(text)


# Parser quality, three ways (exact match alone is too harsh -- one extra field fails it):
#  valid JSON | exact match | VERDICT accuracy = does Z3 reach the right feasible/infeasible
#  answer from the predicted IR? Verdict is what actually matters for the verifier.
from irlib.ir_schema import ir_from_compact
from irlib.constraint_encoder import check_feasibility

def eval_parser(rows, limit=60, parse_fn=None):
    stats = dict(n=0, valid=0, exact=0, verdict_ok=0, verdict_n=0)
    failures = []
    for row in rows[:limit]:
        stats['n'] += 1
        try:
            pred = (parse_fn or parse_intent)(row['query'])
            stats['valid'] += 1
        except Exception as e:
            failures.append((row['query'], 'invalid JSON'))
            continue
        stats['exact'] += pred == row['ir_for_training']
        gold = row['debug_ground_truth']
        if gold['conflict_type'] == 'completion_vs_pause':
            continue  # needs a duration model; Z3 cannot judge these
        # Give device-event anchors the simulator-known time from the gold IR (as the runtime would).
        try:
            for gi, g in enumerate(pred['goals']):
                for ri, r in enumerate(g['readings']):
                    if r['anchor']['kind'] == 'device_event':
                        r['resolved_at_minutes'] = gold['goals'][gi]['readings'][ri].get('resolved_at_minutes')
            verdict = check_feasibility(ir_from_compact(pred, row['query'], gold['base_time'])).feasible
            stats['verdict_n'] += 1
            stats['verdict_ok'] += verdict == (row['meta']['case'] == 'feasible')
            if verdict != (row['meta']['case'] == 'feasible'):
                failures.append((row['query'], f"wrong verdict: predicted {'feasible' if verdict else 'infeasible'}, gold {row['meta']['case']}"))
        except Exception as e:
            failures.append((row['query'], f'bad structure: {type(e).__name__}'))
    return stats, failures

val_stats, val_failures = eval_parser(val_rows)
print(val_stats)
print(f"valid JSON {val_stats['valid']}/{val_stats['n']} | exact {val_stats['exact']}/{val_stats['n']} | "
      f"verdict accuracy {val_stats['verdict_ok']}/{val_stats['verdict_n']}")
for q, why in val_failures[:5]:
    print(' -', why, '|', q[:80])

# Same metric on the HELD-OUT native test episodes (never trained on) -- this is the number to report.
test_rows = load_jsonl('data/qt4_ir_v2_test_held_out.jsonl')
test_stats, test_failures = eval_parser(test_rows, limit=100)
print('HELD-OUT', test_stats)
import json as _json
_json.dump({'val': val_stats, 'test': test_stats}, open('parser_eval.json', 'w'), indent=2)


### Parser ablations: few-shot (no fine-tuning) baseline, and error analysis

**Few-shot baseline.** The *same* Qwen3-1.7B with the LoRA adapter switched off, shown 4 training examples in the prompt. If this scores close to the fine-tuned parser, fine-tuning is not buying much; if it is far below, fine-tuning is justified.

**Error analysis.** Every held-out query the fine-tuned parser got wrong (invalid JSON, bad structure, or the wrong feasible/infeasible verdict) is saved to `parser_errors.json`, so you can read them and describe the failure types in the paper.


In [ ]:
try:
    def fewshot_parse(query, k=4, max_new_tokens=700):
        feas = [r for r in train_rows if r['meta']['case'] == 'feasible'][:k // 2]
        infe = [r for r in train_rows if r['meta']['case'] == 'infeasible'][:k // 2]
        shots = [x for pair in zip(feas, infe) for x in pair]
        prompt = f"<|system|>\n{SYSTEM_PROMPT}\n"
        for r in shots:
            prompt += f"<|user|>\n{r['query']}\n<|assistant|>\n{json.dumps(r['ir_for_training'], ensure_ascii=False)}\n"
        prompt += f"<|user|>\n{query}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(intent_parser.device)
        with intent_parser.disable_adapter():   # the untouched base model
            out = intent_parser.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                         stop_strings=['<|user|>'], tokenizer=tokenizer)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        return json.JSONDecoder().raw_decode(text)[0]   # tolerate trailing text after the JSON object

    fs_stats, fs_failures = eval_parser(test_rows, limit=100, parse_fn=fewshot_parse)
    print('FEW-SHOT (no fine-tuning), held-out:', fs_stats)
    print(f"valid JSON {fs_stats['valid']}/{fs_stats['n']} | verdict accuracy {fs_stats['verdict_ok']}/{fs_stats['verdict_n']}   "
          f"(fine-tuned: {test_stats['valid']}/{test_stats['n']} valid, {test_stats['verdict_ok']}/{test_stats['verdict_n']} verdicts)")

    _pe = json.load(open('parser_eval.json'))
    _pe['fewshot_test'] = fs_stats
    json.dump(_pe, open('parser_eval.json', 'w'), indent=2)

    json.dump({'held_out_failures': [{'query': q, 'why': w} for q, w in test_failures],
               'fewshot_failures_count': len(fs_failures)}, open('parser_errors.json', 'w'), indent=2, ensure_ascii=False)
    print()
    print(f'Fine-tuned parser: {len(test_failures)} held-out problems (saved to parser_errors.json):')
    for q, why in test_failures:
        print(' -', why, '|', q[:110])
except Exception as _e:   # must never abort an unattended run before E1
    import traceback; traceback.print_exc(); print('parser ablation skipped (does not affect E1):', repr(_e)[:300])


## 6. Wire the verified `schedule_workflow` into the ReAct loop

Monkeypatches `TOOL_REGISTRY['schedule_workflow']` and `ReActAgent.run` rather than forking SimuHome's vendored source (CC BY-NC-ND 4.0).


In [ ]:
from irlib.constraint_encoder import check_feasibility
from irlib.ir_schema import ir_from_compact
# MUST be the same module objects SimuHome's evaluator imports (`src.agents...`). Importing them as
# `SimuHome.src.agents...` creates a second copy, and patching that copy changes nothing.
from src.agents.tools import tool_schedule_workflow, TOOL_REGISTRY, run_tool
from src.agents.strategies.react_agent import ReActAgent

def verbalize_rejection(result) -> str:
    return 'I cannot schedule this as stated -- the timing is inconsistent: ' + result.explanation()

def resolve_anchor_minutes(device_id: str) -> float | None:
    resp = run_tool('get_attribute', {
        'device_id': device_id, 'endpoint_id': 1,
        'cluster_id': 'OperationalState', 'attribute_id': 'CountdownTime',
    })
    countdown_seconds = ((resp or {}).get('data') or {}).get('value')
    return None if countdown_seconds is None else float(countdown_seconds) / 60.0

_last_user_query = {'text': None}

import time as _time
VERIFIER_STATS = {'calls': 0, 'no_query': 0, 'parse_fail': 0, 'passed': 0, 'rejected': 0, 'seconds': 0.0}   # seconds = total verify time (parse + Z3)
VERIFIER_LOG = []  # (query, verdict, explanation) for every call -- inspect after the run

def verified_schedule_workflow(args):
    VERIFIER_STATS['calls'] += 1
    query = _last_user_query['text']
    if not query:
        VERIFIER_STATS['no_query'] += 1
        return tool_schedule_workflow(args)
    try:
        ir_dict = parse_intent(query)
    except Exception as e:
        VERIFIER_STATS['parse_fail'] += 1
        VERIFIER_LOG.append((query[:100], 'parse_fail', repr(e)[:100]))
        return tool_schedule_workflow(args)
    now_resp = run_tool('get_current_time', {})
    base_time = (now_resp.get('data') or {}).get('now')
    for goal in ir_dict['goals']:
        for reading in goal['readings']:
            if reading['anchor']['kind'] == 'device_event' and reading.get('resolved_at_minutes') is None:
                reading['resolved_at_minutes'] = resolve_anchor_minutes(reading['anchor']['device_id'])
    ir = ir_from_compact(ir_dict, query, base_time)
    result = check_feasibility(ir)
    VERIFIER_LOG.append((query[:100], 'passed' if result.feasible else 'rejected', result.explanation()[:200]))
    if result.feasible:
        VERIFIER_STATS['passed'] += 1
        return tool_schedule_workflow(args)
    VERIFIER_STATS['rejected'] += 1
    return {
        'status': {'code': 409, 'message': 'INFEASIBLE_SCHEDULE'},
        'data': None,
        'error': {'type': 'TEMPORAL_CONTRADICTION', 'detail': verbalize_rejection(result)},
    }

_verified_impl = verified_schedule_workflow
def verified_schedule_workflow(args):   # same function, timed (the latency overhead the paper should report)
    _t0 = _time.time()
    try:
        return _verified_impl(args)
    finally:
        VERIFIER_STATS['seconds'] += _time.time() - _t0

TOOL_REGISTRY['schedule_workflow'] = verified_schedule_workflow

_original_run = ReActAgent.run
# Prompt-only comparison arm (E1 'selfcheck'): the SAME agent, no Z3, just told to check time consistency itself.
SELF_CHECK = {'on': False}
SELF_CHECK_PROMPT = (' IMPORTANT: before scheduling anything, work out the exact time of every requested action. If two parts of the request'
                     ' give the same event two different times (a contradiction), do NOT call schedule_workflow; call finish and explain the conflict.')
def _patched_run(self, user_query, **kwargs):
    _last_user_query['text'] = user_query            # the verifier parses the ORIGINAL query
    if SELF_CHECK['on']:
        user_query = user_query + SELF_CHECK_PROMPT
    return _original_run(self, user_query, **kwargs)
ReActAgent.run = _patched_run


### Verifier on/off switch

Both E1 arms run in this process (see section 3). `set_verifier(False)` restores SimuHome's original `schedule_workflow` for the baseline arm; `set_verifier(True)` installs the verified one. It starts OFF.


In [ ]:
_orig_schedule = tool_schedule_workflow

def set_verifier(on):
    TOOL_REGISTRY['schedule_workflow'] = verified_schedule_workflow if on else _orig_schedule

def set_selfcheck(on):
    SELF_CHECK['on'] = on

set_verifier(False)
print('verifier is OFF until E1 turns it on for the verified arm')


## 7. Full evaluation (E1) + ablations

Reuse SimuHome's own scoring (`src/pipelines/episode_evaluation/qt4/{feasible,infeasible_1,infeasible_2,infeasible_3}.py`) so numbers are directly comparable to the paper's table. Run on the **held-out native episodes only** (`data/qt4_ir_v2_test_held_out.jsonl` lists exactly which ones — never the synthesized episodes used in fine-tuning).


**Important:** every arm (baseline, selfcheck, verified) runs through `run_inproc` (section 3), which executes SimuHome's evaluator **inside this notebook process**. Never use SimuHome's `eval-start` CLI for E1: it starts a separate process, so the request shim and the verifier patch would silently not apply.


### E1 overnight plan runner

Edit **only the `PLAN` block** below. Each entry is `(sub-type, case, arm)`, run in the listed order, one run at a time:

| arm | what runs |
|---|---|
| `baseline` | the agent alone |
| `selfcheck` | the agent + a prompt-only instruction to check time consistency (no parser, no Z3) |
| `verified` | the agent + parser + Z3 verifier |

Safety features for an unattended run: finished runs are skipped; no new run **starts** once `BUDGET_HOURS` would be exceeded; a run that crashes is logged and skipped; two bad runs in a row (crash or API errors) stop the plan so it does not burn the night on a dead API; everything is saved after every run.


In [ ]:
import json, os, collections, shutil, time
os.chdir(os.path.join(PROJECT_ROOT, 'SimuHome'))

# Seeds come from the held-out file, so E1 only scores episodes the parser never trained on.
held_out = [json.loads(l) for l in open(os.path.join(PROJECT_ROOT, 'data/qt4_ir_v2_test_held_out.jsonl'), encoding='utf-8')]
seeds_by_cell = collections.defaultdict(list)
for row in held_out:
    seeds_by_cell[(row['meta']['query_type'], row['meta']['case'])].append(int(row['meta']['seed']))

# ============================ EDIT THIS BLOCK ONLY ============================
E1_MAX_SEEDS = 10      # episodes per (sub-type, case) per arm (the held-out split has about 10 per cell)
BUDGET_HOURS = 6.5     # never START a run if it could push this cell past this many hours
PLAN = [               # (sub-type, case, arm), in priority order; about 1 hour per run at 10 episodes
    ('qt4-1', 'infeasible', 'selfcheck'),
    ('qt4-2', 'infeasible', 'baseline'),
    ('qt4-2', 'infeasible', 'selfcheck'),
    ('qt4-2', 'infeasible', 'verified'),
    ('qt4-2', 'feasible',   'baseline'),
    ('qt4-2', 'feasible',   'verified'),
    ('qt4-2', 'feasible',   'selfcheck'),
    # Later session (hardest sub-type):
    # ('qt4-3', 'infeasible', 'baseline'), ('qt4-3', 'infeasible', 'selfcheck'), ('qt4-3', 'infeasible', 'verified'),
]
# ==============================================================================
ARMS = {'baseline': (False, False), 'selfcheck': (False, True), 'verified': (True, False)}   # (verifier_on, selfcheck_on)

def seed_spec(qt, case):
    return ','.join(str(s) for s in sorted(seeds_by_cell[(qt, case)])[:E1_MAX_SEEDS])

def run_done(run_id):
    # complete = summary exists, episodes were evaluated, and NO infra errors (quota/429/rejected requests)
    if not os.path.exists(f'experiments/{run_id}/run_summary.json'):
        return False
    agg = aggregate_run(run_id)
    return bool(agg) and all(a['overall']['evaluated_total'] > 0 and a['overall']['infra_errors'] == 0 for a in agg.values())

agg_file = os.path.join(PROJECT_ROOT, 'e1_aggregates.json')
log_file = os.path.join(PROJECT_ROOT, 'e1_run_log.json')
aggregates = json.load(open(agg_file)) if os.path.exists(agg_file) else {}
run_log = json.load(open(log_file)) if os.path.exists(log_file) else []

AGENT_SMOKE_TEST = True   # one real episode through the actual agent before the plan (a few minutes); stops the run early if the model cannot follow the ReAct format
if AGENT_SMOKE_TEST and PLAN:
    set_verifier(False); set_selfcheck(False)
    _seed = (sorted(seeds_by_cell[('qt4-1', 'feasible')]) or [1])[0]
    run_inproc('smoke_agent_check', 'qt4-1', 'feasible', str(_seed))
    _o = list(aggregate_run('smoke_agent_check').values())[0]['overall']
    print(f"agent smoke test: evaluated={_o['evaluated_total']} infra_errors={_o['infra_errors']} schema_errors={_o['schema_errors']} (calls so far {LLM_CALLS['count']})")
    if _o['evaluated_total'] == 0 or _o['infra_errors'] or _o['schema_errors']:
        raise RuntimeError('agent smoke test failed (see the counts above and the evaluator output): the model cannot run the ReAct loop here; change LLM_MODEL or fix the API error before spending hours on E1')

T0, slowest, bad_streak = time.time(), 0.0, 0
for qt, case, system in PLAN:
    run_id = f'e1_{system}_{qt}_{case}'.replace('-', '_')
    if run_done(run_id):
        print('already complete:', run_id); continue
    if not seed_spec(qt, case):
        print('no held-out seeds for', qt, case); continue
    if LLM_CALL_CAP and LLM_CALLS['count'] + 150 > LLM_CALL_CAP:
        print(f'call cap nearly reached ({LLM_CALLS["count"]}/{LLM_CALL_CAP}); stopping before {run_id}'); break
    if (time.time() - T0 + slowest) / 3600 > BUDGET_HOURS:
        print(f'time budget reached ({(time.time() - T0) / 3600:.1f} h used); stopping before {run_id}'); break
    shutil.rmtree(f'experiments/{run_id}', ignore_errors=True)   # an earlier attempt crashed or hit a quota/infra error
    verifier_on, selfcheck_on = ARMS[system]
    set_verifier(verifier_on); set_selfcheck(selfcheck_on)
    calls_before, stats_before, t_run = LLM_CALLS['count'], dict(VERIFIER_STATS), time.time()
    entry = {'run_id': run_id, 'system': system, 'cell': f'{qt}|{case}', 'model': LLM_MODEL, 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
    try:
        run_inproc(run_id, qt, case, seed_spec(qt, case))
        per_model = aggregate_run(run_id)
        aggregates[f'{system}|{qt}|{case}'] = per_model
        infra = 0
        for m, a in per_model.items():
            o = a['overall']; infra += o['infra_errors']
            print(f'{system:9s} {qt} {case:10s} accuracy={o["accuracy"]:.2f} ({o["correct"]}/{o["evaluated_total"]}) infra_errors={o["infra_errors"]} schema_errors={o["schema_errors"]}')
            entry.update(accuracy=o['accuracy'], correct=o['correct'], evaluated=o['evaluated_total'], infra_errors=o['infra_errors'], schema_errors=o['schema_errors'])
        bad_streak = bad_streak + 1 if (infra or entry.get('schema_errors', 0) > 3) else 0
        if infra:
            print('   -> API errors in this run (quota?); it will be redone next time.')
    except Exception as e:
        print('RUN FAILED:', run_id, repr(e)[:300]); entry['error'] = repr(e)[:300]; bad_streak += 1
    entry.update(seconds=round(time.time() - t_run), llm_calls=LLM_CALLS['count'] - calls_before,
                 verifier={k: round(VERIFIER_STATS[k] - stats_before[k], 2) for k in VERIFIER_STATS})
    slowest = max(slowest, entry['seconds'])
    run_log.append(entry)
    print('   ', entry['seconds'] // 60, 'min |', entry['llm_calls'], 'LLM calls | verifier:', entry['verifier'])
    json.dump(aggregates, open(agg_file, 'w'), indent=2)   # saved after EVERY run
    json.dump(run_log, open(log_file, 'w'), indent=2)
    json.dump({'stats': VERIFIER_STATS, 'log': VERIFIER_LOG}, open(os.path.join(PROJECT_ROOT, 'verifier_telemetry.json'), 'w'), indent=2)
    if bad_streak >= 2:
        print('two bad runs in a row (API errors or crashes): stopping the plan. Check the provider message above.'); break
set_verifier(False); set_selfcheck(False)
print(f'plan finished after {(time.time() - T0) / 3600:.1f} h')
os.chdir(PROJECT_ROOT)


`e1_aggregates.json` holds SimuHome's own per-episode aggregate for every run, and `verifier_telemetry.json` shows how many times the verifier parsed, passed, rejected, or failed to parse. Download both: the success-rate table is built from those files, not from `run_summary.json` (whose `totals` only says whether each *model run* finished, which is why the earlier table was all 1.0).


### Results table and download bundle

Prints every finished run and writes `overnight_results.zip` to `/kaggle/working/`. In a *Save & Run All* session the zip appears under the notebook's **Output** tab.


In [ ]:
import csv, zipfile, glob
os.chdir(PROJECT_ROOT)
rows = []
for key, per_model in json.load(open('e1_aggregates.json')).items():
    system, qt, case = key.split('|')
    for m, a in per_model.items():
        o = a['overall']
        rows.append([qt, case, system, o['correct'], o['evaluated_total'], round(o['accuracy'], 3), o['infra_errors'], o['schema_errors']])
rows.sort()
print(f"{'sub-type':8s} {'case':10s} {'arm':10s} correct/n  acc   infra schema")
for r in rows:
    print(f'{r[0]:8s} {r[1]:10s} {r[2]:10s} {r[3]:>3}/{r[4]:<3}   {r[5]:.2f}  {r[6]:>3}   {r[7]:>3}')
with open('e1_summary.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['sub_type', 'case', 'arm', 'correct', 'evaluated', 'accuracy', 'infra_errors', 'schema_errors']); w.writerows(rows)

OUT = '/kaggle/working/overnight_results.zip'
keep = ['e1_aggregates.json', 'e1_run_log.json', 'e1_summary.csv', 'verifier_telemetry.json', 'parser_eval.json', 'parser_errors.json', 'train_log.json',
        'data/qt4_ir_v2_test_held_out.jsonl', 'qwen3_ir_parser_adapter_v2'] + glob.glob('SimuHome/experiments/e1_*') + glob.glob('SimuHome/experiments/baseline_*')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for pth in keep:
        if os.path.isfile(pth):
            z.write(pth)
        elif os.path.isdir(pth):
            for root, _, files in os.walk(pth):
                for fn in files:
                    z.write(os.path.join(root, fn))
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), 'MB')


**Ablations (E2-E6, design once E1 is running):** SLM-only (no Z3), oracle-IR-vs-SLM-IR, verbalizer template-vs-raw-core, per-conflict-type breakdown, verify-step latency overhead. Same held-out set and scoring functions as E1, swap one component per ablation, save each to its own CSV under this notebook's directory.

**When done:** click 'Save Version' so `/kaggle/working` (checkpoints, adapter, `data/`, result CSVs) is versioned — that's what should come back into this project's local directory.
